# 01 · Local Left event producer

Reads local pair configuration, queries the Left source, and writes immutable JSON events. It never connects to Databricks. Sync `RECORDS_ROOT` to the cloud prefix behind the External Volume consumed by Notebook 02.


In [ ]:
import os
from pathlib import Path
CONFIG_PATH = Path(os.environ.get('DTRACK_COUNT_WATCH_CONFIG', 'nbs/table_list.example.json'))
RECORDS_ROOT = Path(os.environ.get('DTRACK_RECORDS_ROOT', 'records/count_watch'))
RUN_DATE = os.environ.get('DTRACK_RUN_DATE') or None
PAIR_NAMES = None
DRY_RUN = False


In [ ]:
import json, uuid
from datetime import UTC, date, datetime
from zoneinfo import ZoneInfo

def load_configs(path):
    paths = sorted(path.glob('*.json')) if path.is_dir() else [path]
    values = []
    for item in paths:
        value = json.loads(item.read_text(encoding='utf-8'))
        values.extend(value if isinstance(value, list) else [value])
    required = {'pair_name','config_version','left_query_variable','right_query_variable','left_conn_detail','right_conn_detail'}
    for value in values:
        if missing := sorted(required - value.keys()):
            raise ValueError(f"{value.get('pair_name', '<unknown>')}: missing {missing}")
    return values

configs = [x for x in load_configs(CONFIG_PATH) if x.get('enabled', True)]
if PAIR_NAMES:
    configs = [x for x in configs if x['pair_name'] in set(PAIR_NAMES)]
print(f'{len(configs)} enabled pair(s):', [x['pair_name'] for x in configs])


In [ ]:
def parse_detail(value):
    detail = json.loads(value) if isinstance(value, str) else dict(value)
    forbidden = {'password','token','secret','dsn'}
    def check(item):
        if isinstance(item, dict):
            if any(str(key).lower() in forbidden for key in item):
                raise ValueError('Connection JSON must contain logical references only')
            for nested in item.values(): check(nested)
        elif isinstance(item, list):
            for nested in item: check(nested)
    check(detail)
    return detail

def connect_left(detail):
    source = detail.get('source', 'oracle').lower()
    if source == 'oracle':
        import oracledb
        name = detail['conn_name']
        return oracledb.connect(user=os.environ[f'{name}_USR'], password=os.environ[f'{name}_PWD'], dsn=os.environ[f'{name}_DSN'])
    if source == 'duckdb':
        import duckdb
        return duckdb.connect(detail['path'], read_only=True)
    raise ValueError(f'Unsupported Left source: {source}')

def left_latest(config):
    detail = parse_detail(config['left_conn_detail'])
    source, kind = detail.get('source','oracle').lower(), detail.get('partition_type','date')
    relation = f"({detail['query']}) source_query" if detail.get('query') else detail['table']
    variable = config['left_query_variable']
    expression = f'TRUNC({variable})' if kind == 'date' and source == 'oracle' else (f'CAST({variable} AS DATE)' if kind == 'date' else variable)
    filters = [detail['where']] if detail.get('where') else []
    where = (' WHERE ' + ' AND '.join(f'({x})' for x in filters)) if filters else ''
    connection = connect_left(detail)
    try:
        cursor = connection.cursor(); cursor.execute(f'SELECT MAX({expression}) FROM {relation}{where}')
        latest = cursor.fetchone()[0]
        if latest is None: raise ValueError('latest Left partition is empty')
        text = str(latest)[:10] if kind == 'date' else str(latest)
        literal = f"DATE '{text}'" if kind == 'date' else (text if kind in {'int','numeric'} else "'" + text.replace("'","''") + "'")
        cursor.execute(f'SELECT COUNT(*) FROM {relation}' + (' WHERE ' if not filters else where + ' AND ') + f'({expression} = {literal})')
        return text, int(cursor.fetchone()[0])
    finally: connection.close()


In [ ]:
def write_event(record):
    folder = RECORDS_ROOT / f"query_date={record['query_date']}" / f"pair_name={record['pair_name']}"
    path = folder / f"{record['check_id']}--{record['event_id']}.json"
    if DRY_RUN: print(json.dumps(record, indent=2)); return path
    folder.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(record, sort_keys=True) + '\n', encoding='utf-8')
    temporary.replace(path); return path

written = []
for config in configs:
    zone = ZoneInfo(config.get('timezone') or 'America/New_York')
    query_date = date.fromisoformat(RUN_DATE) if RUN_DATE else datetime.now(zone).date()
    check_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f"count-watch:{config['pair_name']}:{config['config_version']}:{query_date}"))
    folder = RECORDS_ROOT / f'query_date={query_date}' / f"pair_name={config['pair_name']}"
    prior = [json.loads(p.read_text()) for p in folder.glob(f'{check_id}--*.json')]
    if any(x.get('count_state') == 'PENDING_RIGHT' for x in prior): print('already prepared', check_id); continue
    checked_at = datetime.now(UTC)
    try: left_partition, left_count = left_latest(config); state, error = 'PENDING_RIGHT', None
    except Exception as exc: left_partition, left_count, state, error = None, None, 'ERROR', f'{type(exc).__name__}: {exc}'
    record = {'schema_version':1,'event_id':str(uuid.uuid4()),'check_id':check_id,'pair_name':config['pair_name'],'config_version':int(config['config_version']),'query_date':str(query_date),'timezone':config.get('timezone') or 'America/New_York','review_status':config.get('review_status','NOT_CHECKED'),'offset_days':int(config.get('offset_days') or 0),'aws_name':config.get('aws_name'),'left_query_variable':config['left_query_variable'],'right_query_variable':config['right_query_variable'],'left_conn_detail':json.dumps(parse_detail(config['left_conn_detail']),sort_keys=True),'right_conn_detail':json.dumps(parse_detail(config['right_conn_detail']),sort_keys=True),'left_partition':left_partition,'left_count':left_count,'count_state':state,'left_checked_at':checked_at.isoformat(),'error_message':error,'created_at':checked_at.isoformat(),'updated_at':checked_at.isoformat()}
    written.append(write_event(record))
print(f"{'Prepared' if DRY_RUN else 'Wrote'} {len(written)} immutable event file(s)")
for path in written: print(path)
